In [35]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import load_model
import joblib

loaded_model = load_model('../models/model_bigru.keras')
scaler = joblib.load('../models/scaler_minmax_gru.pkl')

def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:i+sequence_length])
        y.append(data[i+sequence_length])
    return np.array(X), np.array(y)

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(numerator / denominator) * 100

data_folder = '../data/raw/'
sequence_length = 3
prix = {}
dates = {}  # Nouveau dictionnaire pour stocker les dates
metrics_dict = {}
action_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]
for file in action_files:
    action_name = os.path.splitext(file)[0]
    
    # Charger les données avec l'index des dates
    data = pd.read_csv(os.path.join(data_folder, file), index_col='Date', parse_dates=True)['Close']
    
    # Normaliser les données
    scaled_data = scaler.transform(data.values.reshape(-1, 1))
    
    # Créer des séquences
    X, y = create_sequences(scaled_data, sequence_length)
    X = X.reshape((X.shape[0], X.shape[1], 1))
    
    # Faire des prédictions
    predictions = loaded_model.predict(X)
    
    # Revenir à l'échelle originale
    predictions_unscaled = scaler.inverse_transform(predictions)
    y_unscaled = scaler.inverse_transform(y.reshape(-1, 1))
    
    # Stocker les prix prédits avec les dates correspondantes
    # Les dates commencent après la séquence initiale
    dates_array = data.index[sequence_length:]
    prix[action_name] = predictions_unscaled.flatten()
    dates[action_name] = dates_array
    
    # Calcul des métriques
    metrics_dict[action_name] = {
        'MSE': mean_squared_error(y_unscaled, predictions_unscaled),
        'MAE': mean_absolute_error(y_unscaled, predictions_unscaled),
        'R²': r2_score(y_unscaled, predictions_unscaled),
        'SMAPE': smape(y_unscaled, predictions_unscaled)
    }

# Création du DataFrame final avec les dates
prix_predits = pd.DataFrame()
for action_name in prix:
    prix_predits[action_name] = pd.Series(prix[action_name], index=dates[action_name])

# Sauvegarder le DataFrame avec les dates
prix_predits.to_csv('../data/predictions/prix_bigru.csv')

79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 172ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [36]:
for file in action_files:
    action_name = os.path.splitext(file)[0]
    print(f"{action_name} : {metrics_dict[action_name]}")

AAPL : {'MSE': 3.444092041617362, 'MAE': 1.0461657484293456, 'R²': 0.9985465613132066, 'SMAPE': 1.6169679270339181}
ADBE : {'MSE': 245.20053383254267, 'MAE': 8.253869712329056, 'R²': 0.9917598514521123, 'SMAPE': 2.410047864413046}
AMZN : {'MSE': 5.858994671818233, 'MAE': 1.41484785781466, 'R²': 0.9979352369245057, 'SMAPE': 1.828369831113183}
BAC : {'MSE': 0.31018042636116105, 'MAE': 0.38064808475805323, 'R²': 0.9962658945105443, 'SMAPE': 1.8192479050978798}
CMCSA : {'MSE': 0.35998806757327545, 'MAE': 0.416690793426089, 'R²': 0.996022675882451, 'SMAPE': 1.3437163272791963}
DIS : {'MSE': 5.388678223941952, 'MAE': 1.492400256966502, 'R²': 0.9947429951151738, 'SMAPE': 1.3590389161632532}
GOOGL : {'MSE': 2.346387213275163, 'MAE': 0.9349200489506807, 'R²': 0.9980232067378187, 'SMAPE': 1.4713770565806688}
GS : {'MSE': 25.833897101934944, 'MAE': 3.3923945705648917, 'R²': 0.9946275178819605, 'SMAPE': 1.631364578693031}
HD : {'MSE': 16.341978427517002, 'MAE': 2.4519201563082436, 'R²': 0.99770460

In [10]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import load_model
import joblib

loaded_model = load_model('../models/model_bigru.keras')
scaler = joblib.load('../models/scaler_minmax_gru.pkl')

c:\Users\Nael\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [34]:


# --- Exemple de données d'entrée (à remplacer par vos vraies données) ---
# Simulez les 3 jours de données historiques jusqu'au 2 juillet 2025
# Chaque sous-liste représente un jour, et chaque élément la caractéristique (ex: prix de clôture)
# Si vous avez plusieurs caractéristiques (prix, volume, etc.), ajoutez-les ici.
# Assurez-vous que l'ordre des caractéristiques est le même que lors de l'entraînement.
# Ici, je suppose une seule caractéristique (le prix de clôture).
last_3_days_data = np.array([
    [29.66], # Prix de clôture du 30 juin 2025
    [29.40], # Prix de clôture du 1er juillet 2025
    [29.75]  # Prix de clôture du 2 juillet 2025
])

# Prétraiter les données avec le même scaler
# Note : reshape(-1, 1) est important si vous n'avez qu'une seule caractéristique
# et si le scaler s'attend à une entrée 2D (nombre_d_échantillons, nombre_de_caractéristiques)
last_3_days_scaled = scaler.transform(last_3_days_data.reshape(-1, last_3_days_data.shape[1]))

# Remettre en forme pour l'entrée du modèle (1 échantillon, 3 pas de temps, X caractéristiques)
# X sera last_3_days_data.shape[1]
input_for_prediction = last_3_days_scaled.reshape(1, 3, last_3_days_data.shape[1])

# --- Faire la prédiction ---
predicted_scaled_price = loaded_model.predict(input_for_prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


In [35]:
# --- Dénormaliser la prédiction ---
# Si votre scaler a été entraîné sur une seule caractéristique (par exemple, seulement le prix),
# vous pouvez dénormaliser directement.
# Si votre scaler a été entraîné sur plusieurs caractéristiques mais que votre modèle
# ne prédit qu'une seule (le prix), vous devrez créer un tableau temporaire.

# Option 1: Scaler entraîné sur une seule caractéristique (prix)
final_predicted_price = scaler.inverse_transform(predicted_scaled_price)[0][0]

# Option 2: Scaler entraîné sur plusieurs caractéristiques (prix, volume, etc.)
# Supposons que le prix est la 0ème caractéristique.
# dummy_array = np.zeros((predicted_scaled_price.shape[0], scaler.n_features_in_))
# dummy_array[:, 0] = predicted_scaled_price.flatten()
# predicted_denormalized_full = scaler.inverse_transform(dummy_array)
# final_predicted_price = predicted_denormalized_full[0, 0]


print(f"Le prix prédit pour le 3 juillet 2025 est : {final_predicted_price:.2f}")

Le prix prédit pour le 3 juillet 2025 est : 29.60
